# 01. Setup & Data Loading

Multi-Chromatic Spatial Pattern Classification 프로젝트의 핵심 설정 및 데이터 로딩 모듈.

**구성:**
1. 환경 설정 및 임포트
2. Ground Truth & 인접 위상 정의
3. 데이터 로딩 함수 (7가지 Descriptor Vector)
4. 통합 데이터 로딩 실행

## 1. 환경 설정 및 임포트

In [ ]:
import os, glob, gc, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, f1_score,
                             confusion_matrix, silhouette_score,
                             calinski_harabasz_score, davies_bouldin_score)
from sklearn.base import clone
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

# Google Colab 환경 감지
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 경로 설정
BASE_DIR = '/content/drive/MyDrive/URP' if IN_COLAB else '.'
VECTOR_DIR = os.path.join(BASE_DIR, '1224_Vectors')
OUTPUT_DIR = os.path.join(BASE_DIR, 'Final_Results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATA_PATHS = {
    'Inter_PI':       os.path.join(VECTOR_DIR, 'Inter_PI'),
    '3D_PI':          os.path.join(VECTOR_DIR, '3D_PI'),
    'Ord_PI':         os.path.join(VECTOR_DIR, 'Ord_PI'),
    'Sixpack_Chroma': os.path.join(VECTOR_DIR, 'Sixpack_Chroma'),
    'Sixpack_Rips':   os.path.join(VECTOR_DIR, 'Sixpack_Rips'),
}

# 하이퍼파라미터
C_VALUES       = [0.5, 1.0, 2.0]
REDUCTION_DIM  = 20
N_SPLITS       = 5
RANDOM_STATE   = 42

print(f'IN_COLAB={IN_COLAB}')
print(f'VECTOR_DIR={VECTOR_DIR}')

## 2. Ground Truth & 인접 위상 정의

8×8×8 파라미터 공간 (RR, RG, GG) → 512개 시뮬레이션 → 12개 Phase 분류.

**Adjacent Phases**: Phase diagram 상 경계를 공유하는 phase 쌍.  
Soft Accuracy에서 인접 phase 간 오분류를 정답으로 처리.

In [ ]:
# Ground Truth Matrices (8 RG slices × 8 RR rows × 8 GG cols)
M1=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3]]
M2=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,4],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,4,4],[2,2,3,3,3,3,3,3]]
M3=[[6,6,7,7,7,7,7,7],[6,6,6,7,7,7,7,7],[9,6,3,3,3,3,3,3],[9,10,3,4,4,3,3,4],[9,10,3,3,4,4,3,4],[9,10,3,4,4,4,4,4],[9,10,3,4,3,4,4,4],[9,10,3,4,3,4,4,4]]
M4=[[6,6,12,12,7,7,7,7],[6,6,12,12,7,7,7,7],[9,6,6,11,7,7,4,4],[9,9,6,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,4,4,4,4,4],[9,9,10,4,4,4,4,4]]
M5=[[6,6,12,12,12,12,7,7],[6,6,12,12,12,12,12,7],[9,9,6,11,11,11,12,11],[9,9,6,11,11,11,4,4],[9,9,13,13,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,10,10,4,4,4,4]]
M6=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,11],[9,9,6,11,11,11,11,11],[9,9,6,6,6,13,4,4],[9,9,6,13,13,4,4,4],[9,9,6,13,4,4,4,4],[9,9,6,13,4,4,4,4]]
M7=[[6,6,12,12,12,12,12,12],[9,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,12],[9,6,6,11,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,11,11,11,4],[9,9,6,6,13,13,4,4],[9,9,6,13,13,4,4,4]]
M8=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,6,11,11,11,11],[9,6,6,6,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,6,11,11,11],[9,9,6,6,13,13,11,11],[9,9,6,6,13,13,11,4]]
GROUND_TRUTH_M = np.asarray([M1,M2,M3,M4,M5,M6,M7,M8])

def get_label_from_index(task_id):
    """task_id (1-based) → GT label.  idx = RR*64 + RG*8 + GG"""
    idx = task_id - 1
    RR_idx = idx // 64
    RG_idx = (idx % 64) // 8
    GG_idx = idx % 8
    return GROUND_TRUTH_M[RG_idx][RR_idx][GG_idx]

def extract_adjacent_phases(matrices):
    """GT 매트릭스에서 인접 phase 쌍 자동 추출."""
    adj = set()
    for M in matrices:
        M = np.array(M)
        r, c = M.shape
        for i in range(r):
            for j in range(c):
                cur = M[i,j]
                for di,dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                    ni, nj = i+di, j+dj
                    if 0<=ni<r and 0<=nj<c and cur!=M[ni,nj]:
                        adj.add(tuple(sorted([int(cur),int(M[ni,nj])])))
    d = {}
    for p1,p2 in adj:
        d.setdefault(p1,[]).append(p2)
        d.setdefault(p2,[]).append(p1)
    return d

ADJACENT_PHASES = extract_adjacent_phases(GROUND_TRUTH_M)
ALL_CLASSES = sorted(np.unique(GROUND_TRUTH_M))

def soft_accuracy_score(y_true, y_pred, adj=ADJACENT_PHASES):
    """인접 위상 오분류를 정답으로 처리하는 정확도."""
    n = len(y_true)
    correct = sum(1 for t,p in zip(y_true,y_pred)
                  if t==p or (t in adj and p in adj[t]) or (p in adj and t in adj[p]))
    return correct/n if n>0 else 0.0

print(f'Classes: {ALL_CLASSES} ({len(ALL_CLASSES)} classes)')
print(f'Adjacent pairs: {len(ADJACENT_PHASES)} entries')

## 3. 데이터 로딩 함수

7가지 Descriptor Vector:
- **Ord_PI**: Ordinary Persistence Image (VR filtration → PI)
- **Inter_PI**: Mixup Barcode 기반 Interaction PI
- **3D_PI**: Mixup Barcode의 3D PI
- **Sixpack_Rips**: Rips complex 기반 Six-pack (최고 성능)
- **Sixpack_Chroma**: Chromatic Alpha 기반 Six-pack
- **Inter+Ord**: Inter_PI + Ord_PI 결합
- **3D+Ord**: 3D_PI + Ord_PI 결합

In [ ]:
def load_generic_pi(data_dir, prefix):
    """Inter_PI / 3D_PI / Ord_PI 공용 PI 벡터 로더."""
    files = sorted(glob.glob(os.path.join(data_dir, f'{prefix}_*.npz')))
    print(f'  Found {len(files)} files')
    X_list, y_list = [], []
    for fp in files:
        try:
            sim_idx = int(os.path.basename(fp).split('_')[-1].split('.')[0])
            label = get_label_from_index(sim_idx)
            data = np.load(fp, allow_pickle=True)
            features = []
            for key in ('arr_0', 'arr_1'):
                arr = data[key]
                if hasattr(arr, 'item') and arr.ndim == 0: arr = arr.item()
                elif arr.shape == (1,): arr = arr[0]
                if isinstance(arr, dict):
                    for k in sorted(arr.keys()):
                        val = arr[k]
                        if isinstance(val, dict):
                            for dk in sorted(val.keys()):
                                features.extend(np.asarray(val[dk]).flatten())
                        else:
                            features.extend(np.asarray(val).flatten())
                else:
                    features.extend(np.asarray(arr).flatten())
            X_list.append(features)
            y_list.append(label)
        except Exception as e:
            print(f'  Error {fp}: {e}')
    if not X_list:
        return None, None
    return np.nan_to_num(np.array(X_list, dtype=float)), np.array(y_list)


def _extract_stat_features(barcode):
    """barcode → 12-dim 통계 feature 벡터."""
    if len(barcode) == 0:
        return np.zeros(12)
    bc = np.array(barcode)
    if bc.ndim == 1:
        if len(bc) % 2 == 0: bc = bc.reshape(-1, 2)
        elif len(bc) > 2: bc = bc[:len(bc)//2*2].reshape(-1, 2)
        else: bc = np.array([[0., 0.]])
    if bc.ndim == 1 or bc.shape[1] < 2:
        return np.zeros(12)
    ls = bc[:, 1] - bc[:, 0]
    b, d = bc[:, 0], bc[:, 1]
    feats = [len(bc), np.mean(ls), np.std(ls), np.max(ls), np.min(ls),
             np.sum(ls), np.mean(b), np.std(b), np.mean(d), np.std(d),
             np.median(ls)]
    p = ls / np.sum(ls) if np.sum(ls) > 0 else ls
    p = p[p > 0]
    feats.append(-np.sum(p * np.log(p + 1e-10)) if len(p) > 0 else 0)
    return np.array(feats)

BARCODE_TYPES = ['domain', 'codomain', 'relative', 'image', 'kernel', 'cokernel']


def load_sixpack_rips(data_dir, selected_types=None):
    """Sixpack_Rips 로더 → 288D (2방향×6type×2dim×12stat)."""
    if selected_types is None: selected_types = BARCODE_TYPES
    files = sorted(glob.glob(os.path.join(data_dir, 'Sixpack_Rips_*.npz')))
    print(f'  Found {len(files)} files')
    X_list, y_list = [], []
    for fp in files:
        try:
            sim_idx = int(os.path.basename(fp).split('_')[-1].split('.')[0])
            label = get_label_from_index(sim_idx)
            data = np.load(fp, allow_pickle=True)
            sp = {'A_to_B': data['arr_0'].item(), 'B_to_A': data['arr_1'].item()}
            feats = []
            for d_key in ['A_to_B', 'B_to_A']:
                dd = sp[d_key]
                for bt in BARCODE_TYPES:
                    for dim_key in [0, 1]:
                        if bt in selected_types and bt in dd and dim_key in dd[bt]:
                            feats.extend(_extract_stat_features(np.array(dd[bt][dim_key])))
                        elif bt in selected_types:
                            feats.extend(np.zeros(12))
            X_list.append(feats)
            y_list.append(label)
        except Exception as e:
            print(f'  Error {fp}: {e}')
    if not X_list:
        return None, None
    return np.nan_to_num(np.array(X_list)), np.array(y_list)


def load_sixpack_chroma(data_dir):
    """Sixpack_Chroma 로더 (dict 기반 PI 벡터)."""
    files = sorted(glob.glob(os.path.join(data_dir, 'Sixpack_Chroma_*.npz')))
    print(f'  Found {len(files)} files')
    X_list, y_list = [], []
    for fp in files:
        try:
            sim_idx = int(os.path.basename(fp).split('_')[-1].split('.')[0])
            label = get_label_from_index(sim_idx)
            data = np.load(fp, allow_pickle=True)
            features = []
            for key in ('arr_0', 'arr_1'):
                arr = data[key]
                if hasattr(arr, 'item') and arr.ndim == 0: arr = arr.item()
                if isinstance(arr, dict):
                    for k in sorted(arr.keys()):
                        val = arr[k]
                        if isinstance(val, dict):
                            for dk in sorted(val.keys()):
                                features.extend(np.asarray(val[dk]).flatten())
                        else:
                            features.extend(np.asarray(val).flatten())
                else:
                    features.extend(np.asarray(arr).flatten())
            X_list.append(features)
            y_list.append(label)
        except Exception as e:
            print(f'  Error {fp}: {e}')
    if not X_list:
        return None, None
    return np.nan_to_num(np.array(X_list, dtype=float)), np.array(y_list)

print('Data loading functions defined.')

## 4. 통합 데이터 로딩

In [ ]:
def load_all_datasets():
    """모든 데이터셋 로드 + Inter+Ord, 3D+Ord 결합."""
    datasets = {}
    print('=' * 80)
    print('데이터 로딩')
    print('=' * 80)

    for name in ['Inter_PI', '3D_PI', 'Ord_PI']:
        path = DATA_PATHS.get(name)
        if path and os.path.exists(path):
            print(f'\n[{name}]')
            X, y = load_generic_pi(path, name)
            if X is not None:
                datasets[name] = {'X': X, 'y': y}
                print(f'  Shape: {X.shape}, Classes: {len(np.unique(y))}')

    for name, loader in [('Sixpack_Rips', load_sixpack_rips),
                         ('Sixpack_Chroma', load_sixpack_chroma)]:
        path = DATA_PATHS.get(name)
        if path and os.path.exists(path):
            print(f'\n[{name}]')
            X, y = loader(path)
            if X is not None:
                datasets[name] = {'X': X, 'y': y}
                print(f'  Shape: {X.shape}, Classes: {len(np.unique(y))}')

    # 결합 데이터셋
    if 'Inter_PI' in datasets and 'Ord_PI' in datasets:
        datasets['Inter+Ord'] = {
            'X': np.hstack([datasets['Inter_PI']['X'], datasets['Ord_PI']['X']]),
            'y': datasets['Inter_PI']['y'],
        }
        print(f"\n[Inter+Ord] Shape: {datasets['Inter+Ord']['X'].shape}")
    if '3D_PI' in datasets and 'Ord_PI' in datasets:
        datasets['3D+Ord'] = {
            'X': np.hstack([datasets['3D_PI']['X'], datasets['Ord_PI']['X']]),
            'y': datasets['3D_PI']['y'],
        }
        print(f"[3D+Ord] Shape: {datasets['3D+Ord']['X'].shape}")

    return datasets

# === 실행 ===
datasets = load_all_datasets()
METHODS = [m for m in ['Ord_PI','Inter_PI','3D_PI','Sixpack_Rips',
                       'Sixpack_Chroma','Inter+Ord','3D+Ord'] if m in datasets]
print(f'\n로드된 데이터셋: {METHODS}')